In [ ]:
# pip install --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached https://download.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp310-cp310-win_amd64.whl.metadata (28 kB)
  Using cached https://download.pytorch.org/whl/cu124/torchvision-0.21.0%2Bcu124-cp310-cp310-win_amd64.whl.metadata (6.3 kB)
  Using cached https://download.pytorch.org/whl/filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached https://download.pytorch.org/whl/networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached https://download.pytorch.org/whl/fsspec-2025.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached https://download.pytorch.org/whl/sympy-1.13.1-py3-none-any.whl (6.2 MB)
  Using cached https://download.pytorch.org/whl/mpmath-1.3.0-py3-none-any.whl (536 kB)
  Using cached https://download.pytorch.org/whl/numpy-2.1.2-cp310-cp310-win_amd64.whl.metadata (59 kB)
  Using cached https://download.p

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


In [9]:
# pip install onnxruntime-gpu

In [3]:
# pip install insightface

In [4]:
# pip install ultralytics

# Image Retrieval

- This notebook contains code to carry out image retrieval based on the CLIP Image Embeddings generated for the LAION-5B dataset.

## Non-Faiss Exact Cosine Similarity Image Search

This function searches a dataset of precomputed CLIP embeddings to find images most similar to a given text prompt, an image, or a combination of both. It first computes normalized embeddings for the query using the specified CLIP model, then iterates through the dataset stored in Parquet files, loading embeddings in manageable chunks. For each chunk, it calculates cosine similarities between the query and dataset embeddings, combines text and image similarities using a weighted alpha parameter, filters results based on an optional similarity threshold, and collects metadata such as URLs, captions, and original image indices. Finally, it sorts all matches by similarity and returns the top results if requested.

In [ ]:
from transformers import AutoProcessor, AutoModel
from PIL import Image
import torch
from typing import List, Optional
import numpy as np
import pyarrow.dataset as ds
import os
import re
import pyarrow.parquet as pq

def find_similar_images(
    dataset_dir: str,
    model_name: str,
    text_prompt: Optional[str] = None,
    image_path: Optional[str] = None,
    top_n: Optional[int] = None,
    similarity_threshold: Optional[float] = None,
    alpha: float = 0.5
) -> List[dict]:
    ''' 
    Functions used to find images similar to a text or image (or both) based on the CLIP embeddings.

    Inputs:
    - dataset_dir: Directory containing the CLIP embeddings dataset in Parquet format.
    - model_name: Name of the pre-trained CLIP model to use.
    - text_prompt: Optional text prompt to find similar images. (None means only the image is used to search).
    - image_path: Optional path to an image to find similar images. (None means only the text prompt is used to search).
    - top_n: Optional number of top similar images to return. (None means return all).
    - similarity_threshold: Optional threshold for cosine similarity to filter results. (None means no filtering).
    - alpha: Weight for text similarity in the combined similarity score. (0.5 - equal weight between text and image, 1 - text prompt only is used, 0 - image only is used).

    Outputs:
    - List of dictionaries containing URLs, captions, original image indices, and cosine similarities of the most similar images.
    '''

    assert text_prompt or image_path, "You must provide either a text prompt or an image path."

    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    text_embedding = None
    image_embedding = None

    if text_prompt:
        inputs = processor(text=text_prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            text_features = model.get_text_features(**inputs)
        text_embedding = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    if image_path:
        image = Image.open(image_path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        image_embedding = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

    # Convert to numpy for similarity calculation
    text_embedding_np = text_embedding.cpu().squeeze().numpy() if text_embedding is not None else None
    image_embedding_np = image_embedding.cpu().squeeze().numpy() if image_embedding is not None else None

    all_matches = []

    # Loads only matching files from dataset_dir like part-00000.parquet to avoid loading checkpoints
    pattern = re.compile(r"^part-\d{5}\.parquet$")
    valid_files = [
        os.path.join(dataset_dir, f)
        for f in os.listdir(dataset_dir)
        if pattern.match(f)
    ]
    dataset = ds.dataset(valid_files, format="parquet")

    # Iterate through each fragment (.parquet file) composing the dataset
    for fragment in dataset.get_fragments():
        print(f"Processing fragment: {fragment.path}")
        pq_file = pq.ParquetFile(os.path.join(dataset_dir, fragment.path))

        for row_group_index in range(pq_file.num_row_groups):
            table = pq_file.read_row_group(row_group_index)
            df_chunk = table.to_pandas()

            df_chunk = df_chunk.dropna(subset=['embeddings_result'])
            if df_chunk.empty:
                continue

            df_chunk['embeddings_result'] = df_chunk['embeddings_result'].apply(
                lambda x: np.array(x) if isinstance(x, list) else x
            )
            image_embeddings = np.vstack(df_chunk['embeddings_result'].values)

            # Compute similarities separately
            sim_text = np.dot(image_embeddings, text_embedding_np.T) if text_embedding_np is not None else 0
            sim_image = np.dot(image_embeddings, image_embedding_np.T) if image_embedding_np is not None else 0

            # Combine similarity with weights
            combined_sim = None
            if text_embedding_np is not None and image_embedding_np is not None:
                combined_sim = alpha * sim_text + (1 - alpha) * sim_image
            elif text_embedding_np is not None:
                combined_sim = sim_text
            else:
                combined_sim = sim_image

            df_chunk['combined_similarity'] = combined_sim

            if similarity_threshold is not None:
                df_chunk = df_chunk[df_chunk['combined_similarity'] >= similarity_threshold]

            for _, row in df_chunk.iterrows():
                all_matches.append({
                    'url': row.get('url'),
                    'caption': row.get('caption'),
                    'original_image_index': row.get('original_image_index'),
                    'cosine_similarity': row['combined_similarity']
                })

    all_matches.sort(key=lambda x: x['cosine_similarity'], reverse=True)
    top_matches = all_matches[:top_n] if top_n is not None else all_matches

    return top_matches

In [ ]:
dataset_dir = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\clip_embeddings_resumable_symlink\all_images_openai_clip_vit_large_patch14\0000_embeddings'
clip_model_names = ["openai/clip-vit-base-patch16", "openai/clip-vit-base-patch32", "openai/clip-vit-large-patch14"]
model_name = clip_model_names[2] 
text_prompt = "umpire"
search_image_path = r'C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\images\test\TestConstructionWorkerImage.jpg' 
top_n = 10
similarity_threshold = 0.15

image_info_list = find_similar_images(
    dataset_dir=dataset_dir,
    model_name=model_name,
    text_prompt=text_prompt,
    image_path = search_image_path,
    top_n=top_n,
    similarity_threshold=similarity_threshold,
    alpha=1  # 1 = text only, 0 = image only, 0.5 = equal weighting
)

# image_info_list

Processing fragment: C:/MastersRepos/ARI5902-Research-Topics-in-AI/LAION-5B Testing/clip_embeddings_resumable_symlink/all_images_openai_clip_vit_large_patch14/0000_embeddings/part-00000.parquet


In [ ]:
len(image_info_list)

1000

## Faiss Exact Cosine Similarity Image Search

* **`build_faiss_index_with_mapping_resume_exact` function:**
  This function builds or resumes a FAISS index from a list of Parquet files containing precomputed embeddings. It reads embeddings in batches, normalizes them for cosine similarity, and adds them to an exact `IndexFlatIP` FAISS index. The function supports resuming from an existing index and mapping, skipping already indexed embeddings. It maintains a mapping between vector indices and image paths, and periodically saves both the FAISS index and mapping to disk to prevent data loss. It processes large datasets efficiently by iterating over batches and shards, ensuring that even very large embeddings can be indexed without exceeding memory limits.

* **`process_all_prompts_with_resume` function:**
  This function performs an exact similarity search across multiple FAISS shard indexes, derived from the prior function. It computes normalized embeddings for a given text prompt and/or image using a CLIP model, optionally combining them with a weighted alpha parameter. The function then streams through each FAISS shard, performing inner-product searches to retrieve the most similar images, applying an optional similarity threshold. Results from all shards are aggregated, sorted by similarity, and truncated to the top-k matches if requested. This approach allows searching very large embedding collections without loading all data into memory at once.

In [1]:
import os
import json
import gc
from pathlib import Path

import faiss
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
from tqdm import tqdm
import pickle
import ast

# Disable tqdm's background monitor thread to reduce flicker in some environments
tqdm.monitor_interval = 0


def build_faiss_index_with_mapping_resume_exact(
    parquet_files,
    index_path,
    mapping_path,
    save_every_vectors: int = 5_000_000,
    state_path=None,
):
    """
    EXACT FAISS IndexFlatIP builder using CPU with resume + checkpointing.

    - Uses inner product (IP) with L2-normalized vectors => cosine similarity.
    - Supports interruption & resume via JSON state file.
    - Periodically checkpoints FAISS index, mapping, and state using atomic writes.
    - Progress reporting via tqdm (per-parquet-file progress in ROW GROUPS).

    Notes
    -----
    - We process each Parquet file row-group by row-group.
    - Resume is tracked per file in units of row groups completed.
    """

    index_path = str(index_path)
    mapping_path = str(mapping_path)

    if state_path is None:
        state_path = mapping_path + ".state.json"
    state_path = str(state_path)

    parquet_files = [str(f) for f in parquet_files]

    # === Step 1: Load existing FAISS index + mapping (CPU) ===
    if os.path.exists(index_path) and os.path.exists(mapping_path):
        try:
            index = faiss.read_index(index_path)
            with open(mapping_path, "rb") as f:
                idx_to_path = pickle.load(f)
            if not isinstance(idx_to_path, list):
                raise ValueError("mapping_path must contain a list.")
            total_vectors_existing = len(idx_to_path)
        except Exception:
            index = None
            idx_to_path = []
            total_vectors_existing = 0
    else:
        index = None
        idx_to_path = []
        total_vectors_existing = 0

    # === Step 2: Load state (per-file groups processed) ===
    if os.path.exists(state_path):
        with open(state_path, "r", encoding="utf-8") as f:
            state = json.load(f)
        # New key: per_file_groups_processed
        per_file_groups_processed = state.get("per_file_groups_processed", {})
        # Old key: per_file_rows_processed (ignored now for safety)
        # If you really want to attempt to map rows->groups, you'd need per-file metadata.
    else:
        per_file_groups_processed = {}

    # -------------------------------------------
    # Atomic write helper
    # -------------------------------------------
    def atomic_write_bytes(filepath: str, write_fn):
        """
        Atomically write to a file by writing to a temp file, (optionally) fsyncing, and renaming.
        `write_fn(tmp_path)` should fully write the file.
        """
        tmp_path = filepath + ".tmp"

        # Write to temp file
        write_fn(tmp_path)

        # Try to fsync to ensure bytes hit disk (best-effort)
        try:
            with open(tmp_path, "rb") as f:
                try:
                    os.fsync(f.fileno())
                except OSError:
                    # On some systems/drives this can fail; ignore durability-only failure
                    pass
        except FileNotFoundError:
            # If for some reason the temp file isn't there, let os.replace fail below
            pass

        # Atomic replace
        os.replace(tmp_path, filepath)

    # -------------------------------------------
    # CHECKPOINT FUNCTION
    # -------------------------------------------
    vectors_since_last_save = 0

    def save_checkpoint():
        nonlocal index, vectors_since_last_save

        if index is None:
            return

        tqdm.write("Checkpoint: saving index + mapping + state...")

        # 1. Index
        atomic_write_bytes(index_path, lambda tmp: faiss.write_index(index, tmp))

        # 2. Mapping
        def write_mapping(tmp):
            with open(tmp, "wb") as f:
                pickle.dump(idx_to_path, f, protocol=pickle.HIGHEST_PROTOCOL)

        atomic_write_bytes(mapping_path, write_mapping)

        # 3. State (now stores row-group counts)
        state_obj = {
            "per_file_groups_processed": per_file_groups_processed,
            "total_vectors": len(idx_to_path),
        }

        def write_state(tmp):
            with open(tmp, "w", encoding="utf-8") as f:
                json.dump(state_obj, f)

        atomic_write_bytes(state_path, write_state)

        vectors_since_last_save = 0
        tqdm.write("Checkpoint saved.")

    # -------------------------------------------
    # BATCH PARSING FUNCTION
    # -------------------------------------------
    def parse_embeddings_batch(batch: pa.RecordBatch, shard_name: str):
        """
        Returns:
            emb_matrix : np.ndarray | None, shape (n_valid, dim)
            paths      : list[str]
        """
        emb_field = batch["embeddings_result"]
        valid_mask = pc.is_valid(emb_field)
        if pc.sum(valid_mask).as_py() == 0:
            return None, []

        batch_filtered = batch.filter(valid_mask)
        emb_col = batch_filtered["embeddings_result"]
        idx_col = batch_filtered["original_image_index"]

        emb_type = emb_col.type

        try:
            # Fast path: fixed-size list of floats
            if isinstance(emb_type, pa.FixedSizeListType) and pa.types.is_floating(
                emb_type.value_type
            ):
                dim = emb_type.list_size
                flat = emb_col.values.to_numpy(zero_copy_only=False)
                if flat.shape[0] % dim != 0:
                    return None, []

                emb_matrix = flat.reshape(-1, dim).astype("float32", copy=False)
                idx_py = idx_col.to_pylist()
                paths = [f"{shard_name}_images/{i}.jpg" for i in idx_py]
                return emb_matrix, paths

            # Fallback: embeddings stored as lists/strings
            emb_py = emb_col.to_pylist()
            idx_py = idx_col.to_pylist()

            arrs, paths = [], []
            for e, i in zip(emb_py, idx_py):
                if e is None:
                    continue
                if isinstance(e, str):
                    try:
                        e = ast.literal_eval(e)
                    except Exception:
                        continue
                arrs.append(np.asarray(e, dtype="float32"))
                paths.append(f"{shard_name}_images/{i}.jpg")

            if not arrs:
                return None, []
            emb_matrix = np.vstack(arrs).astype("float32", copy=False)
            return emb_matrix, paths

        except Exception:
            return None, []

    # -------------------------------------------
    # MAIN PROCESSING LOOP (PER SHARD)
    # -------------------------------------------
    total_vectors_added_this_run = 0

    for shard_path in parquet_files:
        shard_path = str(shard_path)
        shard_name = Path(shard_path).stem

        # Open Parquet file and determine row groups
        pf = pq.ParquetFile(shard_path)
        num_row_groups = pf.metadata.num_row_groups

        # Number of row-groups already processed for this file (if any)
        groups_done = per_file_groups_processed.get(shard_path, 0)

        # Safety: if file changed (row-group count shrank), restart it
        if groups_done > num_row_groups:
            groups_done = 0

        # Row-group–based progress bar
        pbar = tqdm(
            total=num_row_groups,
            desc=f"Shard {shard_name}",
            unit="group",
            dynamic_ncols=True,
            leave=True,
        )

        # Move bar if resuming
        if groups_done > 0:
            pbar.update(groups_done)

        # Process remaining row groups
        for rg_idx in range(groups_done, num_row_groups):
            table = pf.read_row_group(rg_idx)
            batches = table.to_batches()

            for batch in batches:
                emb, paths = parse_embeddings_batch(batch, shard_name)

                if emb is not None and emb.shape[0] > 0:
                    emb = np.ascontiguousarray(emb, dtype="float32")
                    faiss.normalize_L2(emb)

                    if index is None:
                        dim = emb.shape[1]
                        index = faiss.IndexFlatIP(dim)

                    index.add(emb)
                    idx_to_path.extend(paths)
                    vectors_since_last_save += len(paths)
                    total_vectors_added_this_run += len(paths)

                    # Checkpoint if needed
                    if vectors_since_last_save >= save_every_vectors:
                        save_checkpoint()
                        gc.collect()

            # Mark this row group as completed
            per_file_groups_processed[shard_path] = rg_idx + 1

            # Update progress bar by 1 row group
            pbar.update(1)

        pbar.close()
        del pf
        gc.collect()

    # -------------------------------------------
    # FINAL SAVE
    # -------------------------------------------
    if index is None:
        tqdm.write("No embeddings found; nothing to save.")
        return

    save_checkpoint()

    tqdm.write("Index build complete.")
    tqdm.write(f"Total vectors before run: {total_vectors_existing}")
    tqdm.write(f"New vectors added: {total_vectors_added_this_run}")
    tqdm.write(f"Final total vectors: {len(idx_to_path)}")
    tqdm.write(f"Index saved → {index_path}")
    tqdm.write(f"Mapping saved → {mapping_path}")
    tqdm.write(f"State saved → {state_path}")


# =======================
# DRIVER CODE
# =======================

# List of embedding directories
dataset_dirs = [
    r"F:\Thesis\0000_embeddings_cleaned",
    r"F:\Thesis\0001_embeddings_cleaned",
    r"F:\Thesis\0002_embeddings_cleaned",
    r"F:\Thesis\0003_embeddings_cleaned",
]

output_base_dir = r"G:\Thesis\image_retrieval_faiss_indices"
os.makedirs(output_base_dir, exist_ok=True)

for dataset_dir in dataset_dirs:
    all_files = os.listdir(dataset_dir)
    parquet_files = [
        os.path.join(dataset_dir, f)
        for f in all_files
        if os.path.splitext(f)[1].lower() == ".parquet"
    ]

    dataset_base = Path(dataset_dir).stem

    for parquet_path in parquet_files:
        parquet_name = Path(parquet_path).stem

        # Dynamic output paths
        output_index_path = os.path.join(
            output_base_dir,
            f"faiss_{dataset_base}_{parquet_name}_IndexFlatIP.index",
        )
        output_mapping_path = os.path.join(
            output_base_dir,
            f"faiss_{dataset_base}_{parquet_name}_mapping.pkl",
        )
        output_state_path = os.path.join(
            output_base_dir,
            f"faiss_{dataset_base}_{parquet_name}_state.json",
        )

        print(f"Processing {dataset_base}/{parquet_name}...")

        build_faiss_index_with_mapping_resume_exact(
            parquet_files=[parquet_path],
            index_path=output_index_path,
            mapping_path=output_mapping_path,
            state_path=output_state_path,
            save_every_vectors=100_000,
        )

        print(f"Saved index to {output_index_path} and mapping to {output_mapping_path}\n")

Processing 0000_embeddings_cleaned/part-00000...


Shard part-00000:   0%|          | 0/10 [00:38<?, ?group/s]

Checkpoint: saving index + mapping + state...


Shard part-00000:  10%|█         | 1/10 [01:22<12:21, 82.35s/group]

Checkpoint saved.


...

Checkpoint saved.


Shard part-00008:  50%|█████     | 1/2 [04:07<03:18, 198.83s/group]

Checkpoint: saving index + mapping + state...


Shard part-00008: 100%|██████████| 2/2 [05:28<00:00, 164.11s/group]


Checkpoint saved.
Checkpoint: saving index + mapping + state...
Checkpoint saved.
Index build complete.
Total vectors before run: 0
New vectors added: 265287
Final total vectors: 265287
Index saved → G:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_IndexFlatIP.index
Mapping saved → G:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_mapping.pkl
State saved → G:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_state.json
Saved index to G:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_IndexFlatIP.index and mapping to G:\Thesis\image_retrieval_faiss_indices\faiss_0003_embeddings_cleaned_part-00008_mapping.pkl



In [ ]:
import os
import json
import faiss
import heapq
import shutil
import numpy as np
from transformers import AutoProcessor, AutoModel
import torch
import pickle
from pathlib import Path
from typing import List, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import ctypes
import ctypes.wintypes as wt

# ============================================================
# UTILITY: Chunk list
# ============================================================

def chunk_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


# ============================================================
# UTILITY: Prompt generation
# ============================================================

def generate_custom_list(object_list, template="{object}"):
    return [template.format(object=o) for o in object_list]


# ============================================================
# LOAD PROCESSED PROMPTS FROM JSONL
# ============================================================

def load_processed_prompts_from_jsonl(jsonl_path: str):
    """
    Reads JSONL file line-by-line and extracts existing prompts.
    This is lightweight and resumable.
    """
    processed = set()
    path = Path(jsonl_path)

    if not path.exists():
        return processed

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                p = obj.get("prompt")
                if p:
                    processed.add(p)
            except json.JSONDecodeError:
                print("WARNING: Skipped corrupted JSONL line.")
                continue

    print(f"[Resume] Found {len(processed)} previously processed prompts.")
    return processed


# ============================================================
# APPEND RESULTS TO JSONL
# ============================================================

def append_results_to_jsonl(jsonl_path: str, batch_results: list):
    """
    Append one line per prompt to the JSONL file.
    Atomic per-line write to avoid corruption.
    """
    path = Path(jsonl_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "a", encoding="utf-8") as f:
        for entry in batch_results:
            f.write(json.dumps(entry) + "\n")
            f.flush()
            os.fsync(f.fileno())

    print(f"[JSONL Append] {len(batch_results)} prompts appended → {jsonl_path}")


# ============================================================
# FAISS SEARCH FUNCTION (NO COPYING)
# ============================================================

def search_images(
    shard_index_paths: List[str],
    shard_mapping_paths: List[str],
    model_name: str,
    prompts: List[str],
    negative_prompts: Optional[List[str]] = None,
    top_k: int = 10,
    similarity_threshold: Optional[float] = None,
    device: str = None
):
    """
    Batch FAISS search across shards.
    Does NOT copy images or write any files.
    """

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # Load CLIP model
    processor = AutoProcessor.from_pretrained(model_name, force_download=False)
    model = AutoModel.from_pretrained(model_name, force_download=False).to(device)
    model.eval()

    # Encode prompts
    inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        text_features = model.get_text_features(**inputs)
    text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    # Negative prompts
    if negative_prompts:
        neg_inputs = processor(text=negative_prompts, return_tensors="pt", padding=True,
                               truncation=True).to(device)
        with torch.no_grad():
            neg_features = model.get_text_features(**neg_inputs)
        neg_features = neg_features / neg_features.norm(p=2, dim=-1, keepdim=True)
        neg_mean = neg_features.mean(dim=0, keepdim=True)
        text_features = text_features - neg_mean
        text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

    # Convert to numpy
    query_embs = text_features.cpu().numpy().astype("float32")
    faiss.normalize_L2(query_embs)
    n_queries = query_embs.shape[0]

    # Storage
    all_results = [[] for _ in range(n_queries)]

    # SHARD SEARCH with progress bar
    num_shards = len(shard_index_paths)
    for idx_path, map_path in tqdm(
        list(zip(shard_index_paths, shard_mapping_paths)),
        desc="Searching shards",
        total=num_shards,
        unit="shard"
    ):
        idx_path = Path(idx_path)
        map_path = Path(map_path)

        shard_name = idx_path.stem
        parts = shard_name.split("_")
        group_id = parts[1] if len(parts) > 1 else "unknown"

        index = faiss.read_index(str(idx_path))
        with open(map_path, "rb") as f:
            idx_to_path = pickle.load(f)

        per_shard_k = min(len(idx_to_path), top_k)
        D, I = index.search(query_embs, per_shard_k)

        for qi in range(n_queries):
            for score, idx in zip(D[qi], I[qi]):
                if idx < 0 or idx >= len(idx_to_path):
                    continue
                if similarity_threshold is not None and score < similarity_threshold:
                    continue

                all_results[qi].append({
                    "image_path": idx_to_path[idx],
                    "score": float(score),
                    "shard": shard_name,
                    "group_id": group_id
                })

        del index

    # GLOBAL TOP-K
    final_results = []
    for qi in range(n_queries):
        final_results.append({
            "prompt": prompts[qi],
            "results": heapq.nlargest(top_k, all_results[qi], key=lambda x: x["score"])
        })

    return final_results


# ============================================================
# MAIN PROCESSING LOOP (WITH RESUME)
# ============================================================

def process_all_prompts_with_resume(
    shard_indexes,
    shard_mappings,
    profession_list,
    prompt_templates,
    jsonl_path,
    chunk_size,
    negative_prompts = ["Cartoon", "NSFW", "Sex", "Naked", "Clothing", "Object", "Sign", "Logo"],
    model_name="openai/clip-vit-large-patch14",
    similarity_threshold=0.15,
    top_k=1000
):

    processed = load_processed_prompts_from_jsonl(jsonl_path)

    # Expand profession_list → full prompts
    all_prompts = []
    for p in sorted(set(profession_list)):
        for tmpl in prompt_templates:
            all_prompts.append(tmpl.format(object=p))

    for bidx, chunk in enumerate(chunk_list(all_prompts, chunk_size)):

        # Skip prompts already processed
        to_process = [p for p in chunk if p not in processed]

        if not to_process:
            print(f"[Batch {bidx}] All prompts already processed → Skipping.")
            continue

        print(f"[Batch {bidx}] Processing {len(to_process)} prompts: {to_process}")

        batch_results = search_images(
            shard_index_paths=shard_indexes,
            shard_mapping_paths=shard_mappings,
            model_name=model_name,
            prompts=to_process,
            negative_prompts=negative_prompts,
            top_k=top_k,
            similarity_threshold=similarity_threshold
        )

        # Append
        append_results_to_jsonl(jsonl_path, batch_results)

        # Mark processed
        for r in batch_results:
            processed.add(r["prompt"])

    print("All batches processed.\n")

c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from pathlib import Path
from tqdm import tqdm

faiss_dir = r"G:\Thesis\image_retrieval_faiss_indices"

shard_indexes = sorted([str(p) for p in Path(faiss_dir).glob("faiss_*_IndexFlatIP.index")])
shard_mappings = sorted([str(p) for p in Path(faiss_dir).glob("faiss_*_mapping.pkl")])
assert len(shard_indexes) == len(shard_mappings), "Mismatch between indexes and mappings!"

# List of professions retrieved from papers as well as ChatGPT
profession_list = [
    "Accountant", "Actor", "Actuary", "Administrator", "Air Traffic Controller",
    "Airplane Pilot", "Analyst", "Animal Trainer", "Anthropologist", "Archaeologist",
    "Architect", "Artist", "Astronaut", "Athlete", "Attorney",
    "Auditor", "Baker", "Banker", "Barber", "Barista",
    "Bartender", "Biologist", "Blacksmith", "Content Creator", "Bodyguard",
    "Bounty Hunter", "Brewer", "Bricklayer", "Broker", "Builder",
    "Butcher", "Carpenter", "Caregiver", "Chef", "Chemist",
    "Chiropractor", "Civil Engineer", "Cleaner", "Clerk", "Coach",
    "Comedian", "Composer", "Construction Worker", "Therapist", "Cook",
    "Dancer", "Data Scientist", "Delivery Driver", "Dentist", "Designer",
    "Detective", "Dietitian", "Director", "DJ", "Doctor",
    "Dog Walker", "Driver", "Editor", "Electrician", "Engineer",
    "Entrepreneur", "Farmer", "Fashion Designer", "Firefighter", "Fisherman",
    "Flight Attendant", "Florist", "Gardener", "Geologist", "Graphic Designer",
    "Hairdresser", "Handyman", "Historian", "Hotel Concierge", "Ice Cream Maker",
    "Illustrator", "Translator", "Janitor", "Journalist", "Judge",
    "Laborer", "Lawyer", "Librarian", "Lifeguard", "Logger",
    "Magician", "Makeup Artist", "Marine Biologist", "Mathematician", "Mechanic",
    "Medical Researcher", "Meteorologist", "Midwife", "Miner", "Model",
    "Musician", "News Anchor", "Nurse", "Nutritionist", "Oceanographer",
    "Office Assistant", "Optician", "Painter", "Paramedic", "Park Ranger",
    "Pastry Chef", "Personal Trainer", "Pharmacist", "Photographer", "Physical Therapist",
    "Physicist", "Pilot", "Plumber", "Police Officer", "Politician",
    "Professor", "Software Engineer", "Psychologist", "Realtor", "Researcher",
    "Sailor", "Salesperson", "Scientist", "Screenwriter", "Security Officer",
    "Singer", "Skilled Technician", "Social Worker", "Soldier", "Sound Engineer",
    "Statistician", "Surgeon", "Tailor", "Teacher", "Technician",
    "Veterinarian", "Videographer", "Waiter", "Welder", "Writer",
    "Zoologist","Actuarial Analyst", "Administrative Assistant", "Appraiser", "Archivist", "Art Director",
    "Audio Technician", "Automotive Designer", "Baker Assistant", "Bankruptcy Specialist", "Bioinformatician",
    "Biomedical Engineer", "Brand Manager", "Budget Analyst", "Cartographer", "Chemical Engineer",
    "Urban Planner", "Claims Adjuster", "Clinical Laboratory Scientist", "Compliance Officer", "Conservation Officer",
    "Copywriter", "Court Reporter", "Crime Scene Investigator", "Customer Support Specialist", "Database Administrator",
    "Debt Counselor", "Economist", "Electrical Technician", "Emergency Management Specialist", "Environmental Engineer",
    "Ergonomist", "Estate Planner", "Event Coordinator", "Executive Assistant", "Facilities Manager",
    "Financial Analyst", "Flight Dispatcher", "Forensic Scientist", "Freight Coordinator", "Development Officer",
    "Genetic Counselor", "Grant Writer", "Health Inspector", "Human Resources Specialist", "Industrial Designer",
    "Insurance Underwriter", "Investment Banker", "IT Support Specialist", "Paralegal", "Loan Officer",
    "Logistics Manager", "Market Research Analyst", "Marketing Manager", "Occupational Therapist", "Operations Manager",
    "Payroll Specialist", "Procurement Officer", "Property Manager", "Quality Assurance Inspector", "Roofer"
]

prompt_templates = ["Male {object}", "Female {object}"]

jsonl_path = r"G:\Thesis\ImageRetrieval\Professions\retrieval_results_batchsize_10.jsonl"

process_all_prompts_with_resume(
    shard_indexes=shard_indexes,
    shard_mappings=shard_mappings,
    profession_list=profession_list,
    prompt_templates=prompt_templates,
    jsonl_path=jsonl_path,
    chunk_size=10,
    negative_prompts = ["Cartoon", "NSFW", "Sex", "Naked", "Clothing", "Object", "Sign", "Logo"],
    model_name="openai/clip-vit-large-patch14",
    similarity_threshold=0.15,
    top_k=10_000
)

[Batch 0] Processing 10 prompts: ['Male Accountant', 'Female Accountant', 'Male Actor', 'Female Actor', 'Male Actuarial Analyst', 'Female Actuarial Analyst', 'Male Actuary', 'Female Actuary', 'Male Administrative Assistant', 'Female Administrative Assistant']


c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Searching shards: 100%|██████████| 36/36 [08:02<00:00, 13.39s/shard]


[JSONL Append] 10 prompts appended → G:\Thesis\ImageRetrieval\Professions\retrieval_results_batchsize_10.jsonl
[Batch 1] Processing 10 prompts: ['Male Administrator', 'Female Administrator', 'Male Air Traffic Controller', 'Female Air Traffic Controller', 'Male Airplane Pilot', 'Female Airplane Pilot', 'Male Analyst', 'Female Analyst', 'Male Animal Trainer', 'Female Animal Trainer']


c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Searching shards: 100%|██████████| 36/36 [08:03<00:00, 13.42s/shard]


[JSONL Append] 10 prompts appended → G:\Thesis\ImageRetrieval\Professions\retrieval_results_batchsize_10.jsonl
[Batch 2] Processing 10 prompts: ['Male Anthropologist', 'Female Anthropologist', 'Male Appraiser', 'Female Appraiser', 'Male Archaeologist', 'Female Archaeologist', 'Male Architect', 'Female Architect', 'Male Archivist', 'Female Archivist']


...

[JSONL Append] 10 prompts appended → G:\Thesis\ImageRetrieval\Professions\retrieval_results_batchsize_10.jsonl
[Batch 39] Processing 10 prompts: ['Male Veterinarian', 'Female Veterinarian', 'Male Videographer', 'Female Videographer', 'Male Waiter', 'Female Waiter', 'Male Welder', 'Female Welder', 'Male Writer', 'Female Writer']


Searching shards: 100%|██████████| 36/36 [08:16<00:00, 13.80s/shard]


[JSONL Append] 10 prompts appended → G:\Thesis\ImageRetrieval\Professions\retrieval_results_batchsize_10.jsonl
[Batch 40] Processing 4 prompts: ['Male Zoologist', 'Female Zoologist', 'Male test', 'Female test']


Searching shards: 100%|██████████| 36/36 [08:24<00:00, 14.02s/shard]


[JSONL Append] 4 prompts appended → G:\Thesis\ImageRetrieval\Professions\retrieval_results_batchsize_10.jsonl
All batches processed.



In [3]:
import ctypes
import ctypes.wintypes as wt
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from pathlib import Path
import json
import os

# Windows API binding for CopyFileExW (fast, kernel-level copy)
CopyFileExW = ctypes.windll.kernel32.CopyFileExW

COPY_FILE_RESTARTABLE = 0x00000002  # Similar to Robocopy logic

LPBOOL = ctypes.POINTER(wt.BOOL)

CopyFileExW.argtypes = [
    wt.LPCWSTR,  # existing file
    wt.LPCWSTR,  # new file
    wt.LPVOID,   # callback
    wt.LPVOID,   # callback data
    LPBOOL,      # cancel flag
    wt.DWORD     # copy flags
]
CopyFileExW.restype = wt.BOOL

def copy_results_global(all_results, image_dir, output_dir, max_workers=4):
    """
    High-performance file copy using Windows CopyFileExW.
    Atomic, safe, and as fast as Robocopy.
    """

    print("\n=== GLOBAL IMAGE COPY (CopyFileExW) START ===")

    src_root = Path(image_dir)
    out_root = Path(output_dir)
    out_root.mkdir(parents=True, exist_ok=True)

    max_workers = max(1, min(max_workers, 8))

    # Resolve the correct file path from raw mapping paths
    def resolve_src(r):
        raw = Path(r["image_path"])
        if raw.is_absolute():
            return raw
        # expects structure: root/<group>_images/<image>.jpg
        return src_root / raw.parent.parent / f"{r['group_id']}_images" / raw.name

    # Core Windows ultra-fast copy
    def win_copy(src: Path, dst: Path):
        tmp = dst.with_suffix(dst.suffix + ".tmp")
        dst.parent.mkdir(parents=True, exist_ok=True)

        cancel_flag = wt.BOOL(False)

        ok = CopyFileExW(
            str(src),      # source path (UTF-16)
            str(tmp),      # temp destination
            None,          # callback (unused)
            None,
            ctypes.byref(cancel_flag),
            COPY_FILE_RESTARTABLE
        )

        if not ok:
            raise OSError(f"CopyFileExW failed for {src} → {dst}")

        # Atomic replace
        tmp.replace(dst)

    # Flatten all prompts + images
    items = []
    for entry in all_results:
        pdir = out_root / entry["prompt"].replace(" ", "_")
        for r in entry["results"]:
            items.append((pdir, r))

    # Multi-thread copy
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = []
        with tqdm(total=len(items), desc="Copying images", unit="img") as pbar:
            for pdir, r in items:
                src = resolve_src(r)
                dst = pdir / f"{r['score']:.3f}_{r['group_id']}_{src.name}"
                futures.append(ex.submit(win_copy, src, dst))

            for f in as_completed(futures):
                try:
                    f.result()
                except Exception as e:
                    print("Copy Error:", e)
                pbar.update(1)

    print("=== GLOBAL IMAGE COPY COMPLETE ===\n")

def copy_results_from_jsonl(jsonl_path, image_dir, output_dir, max_workers=4):
    """
    Reads JSONL and copies images using the high-performance CopyFileExW pipeline.
    """

    all_results = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                all_results.append(obj)
            except:
                print("WARNING: Corrupt JSONL entry skipped.")

    return copy_results_global(
        all_results=all_results,
        image_dir=image_dir,
        output_dir=output_dir,
        max_workers=max_workers
    )

jsonl_path = r"G:\Thesis\ImageRetrieval\Professions\retrieval_results_batchsize_10.jsonl"

copy_results_from_jsonl(
    jsonl_path=jsonl_path,
    image_dir=r"G:\Thesis",
    output_dir=r"G:\Thesis\ImageRetrieval\Professions",
    max_workers=32
)


=== GLOBAL IMAGE COPY (CopyFileExW) START ===


Copying images:   1%|          | 16736/3155050 [02:08<4:42:16, 185.29img/s] 

Copy Error: CopyFileExW failed for G:\Thesis\0001_images\16197748.jpg → G:\Thesis\ImageRetrieval\Professions\Male_Accountant\0.182_0001_16197748.jpg


Copying images:   4%|▎         | 112386/3155050 [03:04<28:26, 1783.06img/s]

Copy Error: CopyFileExW failed for G:\Thesis\0001_images\16155264.jpg → G:\Thesis\ImageRetrieval\Professions\Female_Administrator\0.188_0001_16155264.jpg


...

Copy Error: CopyFileExW failed for G:\Thesis\0001_images\16342312.jpg → G:\Thesis\ImageRetrieval\Professions\Male_test\0.169_0001_16342312.jpg


Copying images: 100%|█████████▉| 3148576/3155050 [35:12<00:03, 1654.05img/s]

Copy Error: CopyFileExW failed for G:\Thesis\0001_images\16342312.jpg → G:\Thesis\ImageRetrieval\Professions\Female_test\0.196_0001_16342312.jpg


Copying images: 100%|██████████| 3155050/3155050 [35:16<00:00, 1490.73img/s]

=== GLOBAL IMAGE COPY COMPLETE ===



## Image Cleaning

- This section utilises the Facial Detection Model outlined in the `DatasetAnnotation` directory to filter out images which contain no detectable faces, to ensure we only process valid images. Furthermore, manual checks to more accuratly remove invalid images are to be done following this method, this serves as a good starting point to whittle down the number of images requiring manual validation.

`NOTE:` As of now checks have not been made to determine the best overall model hence both the Yolo & SCRFD models are utilised simultaneously to check for visible faces, once evalution of these models is done the non-optimal model can be removed and its results ignored.

SCRFD: https://github.com/deepinsight/insightface/tree/master

Yolo-Face: https://github.com/YapaLab/yolo-face

In [3]:
import os
import json
import shutil
from pathlib import Path
from tqdm import tqdm
import numpy as np
import time
from ultralytics import YOLO
from insightface.app import FaceAnalysis
from PIL import Image

# MODELS
def load_yolo(model_path, device=0):
    model = YOLO(model_path)
    model.fuse()
    model.to(device)
    return model

def run_yolo_face(model, image_np, conf_thres):
    """Runs YOLO-Face on a single image."""
    result = model([image_np], conf=conf_thres, verbose=False)[0]

    boxes, scores = [], []
    if result.boxes is not None:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            boxes.append([int(x1), int(y1), int(x2 - x1), int(y2 - y1)])
            scores.append(float(box.conf.cpu()))

    return boxes, scores


def load_scrfd(conf_threshold=0.5, ctx_id=0):
    app = FaceAnalysis(allowed_modules=['detection'])
    app.prepare(ctx_id=ctx_id, det_thresh=conf_threshold)
    return app

def run_scrfd(app, image_np, conf_threshold):
    faces = app.get(image_np)

    boxes, scores = [], []
    for face in faces:
        if face.det_score >= conf_threshold:
            x1, y1, x2, y2 = face.bbox.astype(int)
            boxes.append([int(x1), int(y1), int(x2 - x1), int(y2 - y1)])
            scores.append(float(face.det_score))

    return boxes, scores

In [4]:
from typing import Literal

# DIRECTORY PROCESSING LOGIC
def create_output_structure(output_root, rel_subdir):
    """Ensure the output directory for the subfolder exists."""
    out_dir = Path(output_root) / rel_subdir
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir

def process_image_file(filepath, yolo_model, scrfd_model, conf_thresh):
    """Runs both detectors on a single image file."""
    try:
        img = Image.open(filepath).convert("RGB")
        img_np = np.array(img)
    except Exception:
        return None, None, None  # unreadable image

    yolo_boxes, yolo_scores = run_yolo_face(yolo_model, img_np, conf_thresh)
    scrfd_boxes, scrfd_scores = run_scrfd(scrfd_model, img_np, conf_thresh)

    # a face is detected if *either* model sees one
    detected = (len(yolo_boxes) > 0) or (len(scrfd_boxes) > 0)

    return detected, (yolo_boxes, yolo_scores), (scrfd_boxes, scrfd_scores)

def is_subdir_processed(out_subdir):
    """Return True if both annotation files exist and their lengths match the number of images."""
    yolo_json = out_subdir / "yolo_annotations.json"
    scrfd_json = out_subdir / "scrfd_annotations.json"

    if not yolo_json.exists() or not scrfd_json.exists():
        return False

    # Count images
    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    images = [p for p in out_subdir.iterdir() if p.suffix.lower() in img_exts]

    try:
        with open(yolo_json, "r") as f:
            yolo_data = json.load(f)
        with open(scrfd_json, "r") as f:
            scrfd_data = json.load(f)
    except Exception:
        return False

    # Check counts
    return (
        isinstance(yolo_data, list)
        and isinstance(scrfd_data, list)
        and len(yolo_data) == len(images)
        and len(scrfd_data) == len(images)
    )

# MAIN PIPELINE
def process_dataset_dir(
    input_root,
    output_root,
    yolo_model_path,
    confidence_threshold=0.5,
    device=0,
    scrfd_ctx_id=0,
    mode: Literal["copy", "move"] = "copy"
):
    """
    Walks the dataset directory recursively and:
    - runs YOLO-Face and SCRFD on each image
    - copies or moves images if a face was detected
    - preserves subdirectory structure
    - writes annotation files per subdirectory
    - SKIPS subdirectories already fully processed
    """

    # Strict argument validation
    if mode not in ("copy", "move"):
        raise ValueError(f"Invalid mode '{mode}'. Must be 'copy' or 'move'.")

    input_root = Path(input_root)
    output_root = Path(output_root)
    output_root.mkdir(exist_ok=True, parents=True)

    print("Loading YOLO-Face model...")
    yolo_model = load_yolo(yolo_model_path, device=device)

    print("Loading SCRFD model...")
    scrfd_model = load_scrfd(conf_threshold=confidence_threshold, ctx_id=scrfd_ctx_id)

    annotations_yolo = {}
    annotations_scrfd = {}

    # List all images
    all_images = list(input_root.rglob("*.*"))
    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    for img_path in tqdm(all_images, desc="Processing Images"):
        if img_path.suffix.lower() not in img_exts:
            continue

        # Determine output subdirectory
        rel_subdir = img_path.parent.relative_to(input_root)
        rel_subdir_str = str(rel_subdir)
        out_subfolder = Path(output_root) / rel_subdir_str

        # Resume logic: skip subdir if fully processed
        if out_subfolder.exists() and is_subdir_processed(out_subfolder):
            # Fully processed -> skip this subdirectory
            continue

        # Ensure annotation structures ready
        if rel_subdir_str not in annotations_yolo:
            annotations_yolo[rel_subdir_str] = []
            annotations_scrfd[rel_subdir_str] = []

        # Run detectors
        detected, (yolo_boxes, yolo_scores), (scrfd_boxes, scrfd_scores) = \
            process_image_file(img_path, yolo_model, scrfd_model, confidence_threshold)

        if detected:
            out_subfolder.mkdir(parents=True, exist_ok=True)
            dst_path = out_subfolder / img_path.name

            if mode == "copy":
                shutil.copy2(img_path, dst_path)
            else:  # move
                shutil.move(img_path, dst_path)

            # Record annotation
            annotations_yolo[rel_subdir_str].append({
                "image": img_path.name,
                "boxes": yolo_boxes,
                "scores": yolo_scores
            })

            annotations_scrfd[rel_subdir_str].append({
                "image": img_path.name,
                "boxes": scrfd_boxes,
                "scores": scrfd_scores
            })

    # Write annotations for all processed dirs
    print("Writing annotations...")

    for subdir in annotations_yolo:
        out_subfolder = Path(output_root) / subdir
        out_subfolder.mkdir(parents=True, exist_ok=True)

        with open(out_subfolder / "yolo_annotations.json", "w") as f:
            json.dump(annotations_yolo[subdir], f, indent=2)

        with open(out_subfolder / "scrfd_annotations.json", "w") as f:
            json.dump(annotations_scrfd[subdir], f, indent=2)

    print("Done.")
    
################################################################################
# USAGE EXAMPLE
process_dataset_dir(
    input_root=r"G:\Thesis\ImageRetrieval\Professions",
    output_root=r"G:\Thesis\ImageRetrieval\ProfessionsCleaned",
    yolo_model_path="yolov12l-face.pt",
    confidence_threshold=0.5, # Set to 0.5 since this is just a cleaning step the value isn't as important
    device=0,
    scrfd_ctx_id=0,
    mode='copy' # 'move'
)

Loading YOLO-Face model...
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
Loading SCRFD model...
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
model ignore: C:\Users\User/.insightface\mode

Processing Images:   1%|          | 21267/3134949 [19:09<46:44:07, 18.51it/s]


KeyboardInterrupt: 

In [ ]:
import os
import json
import shutil
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from PIL import Image

from ultralytics import YOLO
from insightface.app import FaceAnalysis
from typing import Literal


################################################################################
# MODELS
################################################################################

def load_yolo(model_path, device=0):
    model = YOLO(model_path)
    model.fuse()
    model.to(device)
    return model


def run_yolo_batch(model, batch_imgs, conf_thres):
    """Batch YOLO inference (true GPU batching)."""
    results = model(batch_imgs, conf=conf_thres, verbose=False)
    outputs = []

    for result in results:
        boxes, scores = [], []
        if result.boxes is not None:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                boxes.append([int(x1), int(y1), int(x2 - x1), int(y2 - y1)])
                scores.append(float(box.conf.cpu()))
        outputs.append((boxes, scores))

    return outputs


def load_scrfd(conf_threshold=0.5, ctx_id=0):
    app = FaceAnalysis(allowed_modules=['detection'])
    app.prepare(ctx_id=ctx_id, det_thresh=conf_threshold)
    return app


def run_scrfd_single(scrfd, img_np, conf_threshold):
    faces = scrfd.get(img_np)
    boxes, scores = [], []
    for face in faces:
        if face.det_score >= conf_threshold:
            x1, y1, x2, y2 = face.bbox.astype(int)
            boxes.append([int(x1), int(y1), int(x2 - x1), int(y2 - y1)])
            scores.append(float(face.det_score))
    return boxes, scores


def run_scrfd_parallel(scrfd, batch_imgs, conf_threshold, workers=8):
    """Threaded per-image SCRFD inference."""
    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = list(executor.map(
            lambda im: run_scrfd_single(scrfd, im, conf_threshold),
            batch_imgs
        ))
    return results


################################################################################
# RESUME CHECK
################################################################################

def is_subdir_processed(out_subdir):
    """Return True if both annotation files exist and counts match the number of images."""
    yolo_json = out_subdir / "yolo_annotations.json"
    scrfd_json = out_subdir / "scrfd_annotations.json"

    if not yolo_json.exists() or not scrfd_json.exists():
        return False

    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    images = [p for p in out_subdir.iterdir() if p.suffix.lower() in img_exts]

    try:
        with open(yolo_json, "r") as f:
            yolo_data = json.load(f)
        with open(scrfd_json, "r") as f:
            scrfd_data = json.load(f)
    except Exception:
        return False

    return (
        isinstance(yolo_data, list)
        and isinstance(scrfd_data, list)
        and len(yolo_data) == len(images)
        and len(scrfd_data) == len(images)
    )


################################################################################
# MAIN PIPELINE
################################################################################

def process_dataset_dir(
    input_root,
    output_root,
    yolo_model_path,
    confidence_threshold=0.5,
    device=0,
    scrfd_ctx_id=0,
    mode: Literal["copy", "move"] = "copy",
    batch_size: int = 16,
    scrfd_workers: int = 8
):
    """
    Ultra-fast batched pipeline:
    - YOLO runs in batches on GPU
    - SCRFD runs in CPU thread pool
    - Resume logic enabled
    - Copy/move modes supported
    - Writes JSON after each subdirectory finishes
    """

    # strict validation
    if mode not in ("copy", "move"):
        raise ValueError("mode must be 'copy' or 'move'")

    input_root = Path(input_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    print("Loading YOLO-Face model...")
    yolo_model = load_yolo(yolo_model_path, device=device)

    print("Loading SCRFD model...")
    scrfd_model = load_scrfd(conf_threshold=confidence_threshold, ctx_id=scrfd_ctx_id)

    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    all_images = [p for p in input_root.rglob("*.*") if p.suffix.lower() in img_exts]

    # Group by subdirectory
    subdir_map = {}
    for img in all_images:
        rel = str(img.parent.relative_to(input_root))
        subdir_map.setdefault(rel, []).append(img)

    # Process per subdirectory
    for rel_subdir, img_paths in tqdm(subdir_map.items(), desc="Processing Subdirectories"):
        out_subfolder = output_root / rel_subdir

        # Resume: skip already fully processed subfolder
        if out_subfolder.exists() and is_subdir_processed(out_subfolder):
            continue

        out_subfolder.mkdir(parents=True, exist_ok=True)

        yolo_annotations = []
        scrfd_annotations = []

        # Batch processing
        for i in range(0, len(img_paths), batch_size):
            batch_paths = img_paths[i:i+batch_size]

            # Load batch
            batch_imgs = []
            valid_idx = []

            for idx, img_path in enumerate(batch_paths):
                try:
                    arr = np.array(Image.open(img_path).convert("RGB"))
                    batch_imgs.append(arr)
                    valid_idx.append(idx)
                except:
                    batch_imgs.append(None)

            batch_imgs_clean = [batch_imgs[k] for k in valid_idx]
            if not batch_imgs_clean:
                continue

            # YOLO batch inference
            yolo_results = run_yolo_batch(yolo_model, batch_imgs_clean, confidence_threshold)

            # SCRFD parallel inference
            scrfd_results = run_scrfd_parallel(scrfd_model, batch_imgs_clean,
                                               confidence_threshold, scrfd_workers)

            # Assign outputs
            for local_idx, global_idx in enumerate(valid_idx):
                img_path = batch_paths[global_idx]
                fname = img_path.name

                yolo_boxes, yolo_scores = yolo_results[local_idx]
                scrfd_boxes, scrfd_scores = scrfd_results[local_idx]

                if len(yolo_boxes) > 0 or len(scrfd_boxes) > 0:
                    dst = out_subfolder / fname
                    if mode == "copy":
                        shutil.copy2(img_path, dst)
                    else:
                        shutil.move(img_path, dst)

                    yolo_annotations.append({
                        "image": fname,
                        "boxes": yolo_boxes,
                        "scores": yolo_scores
                    })
                    scrfd_annotations.append({
                        "image": fname,
                        "boxes": scrfd_boxes,
                        "scores": scrfd_scores
                    })

        # **Write JSON immediately after finishing this directory**
        with open(out_subfolder / "yolo_annotations.json", "w") as f:
            json.dump(yolo_annotations, f, indent=2)

        with open(out_subfolder / "scrfd_annotations.json", "w") as f:
            json.dump(scrfd_annotations, f, indent=2)

    print("Done.")


################################################################################
# USAGE EXAMPLE
process_dataset_dir(
    input_root=r"G:\Thesis\ImageRetrieval\Professions",
    output_root=r"G:\Thesis\ImageRetrieval\ProfessionsCleaned",
    yolo_model_path="yolov12l-face.pt",
    confidence_threshold=0.5, # Set to 0.5 since this is just a cleaning step the value isn't as important
    device=0,
    scrfd_ctx_id=0,
    mode='copy', # 'move'
    batch_size = 16,
    scrfd_workers = 16
)

Loading YOLO-Face model...
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
Loading SCRFD model...
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
model ignore: C:\Users\User/.insightface\mode

Processing Subdirectories:   2%|▏         | 8/402 [45:11<34:07:45, 311.84s/it]

In [ ]:
import os
import json
import shutil
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from PIL import Image

from ultralytics import YOLO
from insightface.app import FaceAnalysis
from typing import Literal


################################################################################
# MODELS
################################################################################

def load_yolo(model_path, device=0):
    model = YOLO(model_path)
    model.fuse()
    model.to(device)
    return model


def run_yolo_batch(model, batch_imgs, conf_thres):
    """Batch YOLO inference (true GPU batching)."""
    results = model(batch_imgs, conf=conf_thres, verbose=False)
    outputs = []

    for result in results:
        boxes, scores = [], []
        if result.boxes is not None:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                boxes.append([int(x1), int(y1), int(x2 - x1), int(y2 - y1)])
                scores.append(float(box.conf.cpu()))
        outputs.append((boxes, scores))

    return outputs


def load_scrfd(conf_threshold=0.5, ctx_id=0, scrfd_onnx_path: str | None = None):
    """
    Load SCRFD detector.

    For now this still uses InsightFace's FaceAnalysis wrapper, which already uses
    ONNXRuntime under the hood (GPU if onnxruntime-gpu is installed and ctx_id >= 0).

    The scrfd_onnx_path parameter is included so you can later swap in a custom
    ONNXRuntime-based implementation using the same model file.
    """
    app = FaceAnalysis(allowed_modules=['detection'])
    app.prepare(ctx_id=ctx_id, det_thresh=conf_threshold)
    # NOTE: if you want to confirm the underlying ONNX model, you can inspect:
    # app.models['detection'].model_file
    return app


def run_scrfd_single(scrfd, img_np, conf_threshold):
    faces = scrfd.get(img_np)
    boxes, scores = [], []
    for face in faces:
        if face.det_score >= conf_threshold:
            x1, y1, x2, y2 = face.bbox.astype(int)
            boxes.append([int(x1), int(y1), int(x2 - x1), int(y2 - y1)])
            scores.append(float(face.det_score))
    return boxes, scores


def run_scrfd_parallel(scrfd, batch_imgs, conf_threshold, executor: ThreadPoolExecutor):
    """
    Threaded per-image SCRFD inference using a *persistent* executor.

    This avoids the overhead of creating/destroying a ThreadPoolExecutor on
    every batch, which matters a lot at 3M images.
    """
    futures = [executor.submit(run_scrfd_single, scrfd, img, conf_threshold)
               for img in batch_imgs]
    results = [f.result() for f in futures]
    return results


################################################################################
# RESUME CHECK
################################################################################

def is_subdir_processed(out_subfolder: Path) -> bool:
    """
    Return True if both annotation files exist and counts match the number of images.
    """
    yolo_json = out_subfolder / "yolo_annotations.json"
    scrfd_json = out_subfolder / "scrfd_annotations.json"

    if not yolo_json.exists() or not scrfd_json.exists():
        return False

    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    images = [p for p in out_subfolder.iterdir() if p.suffix.lower() in img_exts]

    try:
        with open(yolo_json, "r") as f:
            yolo_data = json.load(f)
        with open(scrfd_json, "r") as f:
            scrfd_data = json.load(f)
    except Exception:
        return False

    return (
        isinstance(yolo_data, list)
        and isinstance(scrfd_data, list)
        and len(yolo_data) == len(images)
        and len(scrfd_data) == len(images)
    )


################################################################################
# MAIN PIPELINE
################################################################################

def process_dataset_dir(
    input_root,
    output_root,
    yolo_model_path,
    confidence_threshold=0.5,
    device=0,
    scrfd_ctx_id=0,
    mode: Literal["copy", "move"] = "copy",
    batch_size: int = 16,
    scrfd_workers: int = 8,
    scrfd_onnx_path: str | None = None,  # for future custom ONNX-based SCRFD
):
    """
    Ultra-fast batched pipeline:
    - YOLO runs in batches on GPU
    - SCRFD runs in a persistent CPU thread pool
    - Resume logic enabled
    - Copy/move modes supported
    - Writes JSON after each subdirectory finishes
    """

    if mode not in ("copy", "move"):
        raise ValueError("mode must be 'copy' or 'move'")

    input_root = Path(input_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    print("Loading YOLO-Face model...")
    yolo_model = load_yolo(yolo_model_path, device=device)

    print("Loading SCRFD model...")
    scrfd_model = load_scrfd(
        conf_threshold=confidence_threshold,
        ctx_id=scrfd_ctx_id,
        scrfd_onnx_path=scrfd_onnx_path,
    )

    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    all_images = [p for p in input_root.rglob("*.*") if p.suffix.lower() in img_exts]

    # Group images by subdirectory
    subdir_map: dict[str, list[Path]] = {}
    for img in all_images:
        rel = str(img.parent.relative_to(input_root))
        subdir_map.setdefault(rel, []).append(img)

    # Persistent thread pool for SCRFD
    with ThreadPoolExecutor(max_workers=scrfd_workers) as scrfd_executor:
        # Process per subdirectory
        for rel_subdir, img_paths in tqdm(subdir_map.items(), desc="Processing Subdirectories"):
            out_subfolder = output_root / rel_subdir

            # Resume: skip already fully processed subfolder
            if out_subfolder.exists() and is_subdir_processed(out_subfolder):
                continue

            out_subfolder.mkdir(parents=True, exist_ok=True)

            yolo_annotations = []
            scrfd_annotations = []

            # Batch processing
            for i in range(0, len(img_paths), batch_size):
                batch_paths = img_paths[i:i+batch_size]

                # Load batch
                batch_imgs = []
                valid_idx = []

                for idx, img_path in enumerate(batch_paths):
                    try:
                        arr = np.array(Image.open(img_path).convert("RGB"))
                        batch_imgs.append(arr)
                        valid_idx.append(idx)
                    except Exception:
                        batch_imgs.append(None)

                batch_imgs_clean = [batch_imgs[k] for k in valid_idx]
                if not batch_imgs_clean:
                    continue

                # YOLO batch inference
                yolo_results = run_yolo_batch(
                    yolo_model,
                    batch_imgs_clean,
                    confidence_threshold
                )

                # SCRFD threaded inference (reusing one executor)
                scrfd_results = run_scrfd_parallel(
                    scrfd_model,
                    batch_imgs_clean,
                    confidence_threshold,
                    scrfd_executor
                )

                # Assign outputs
                for local_idx, global_idx in enumerate(valid_idx):
                    img_path = batch_paths[global_idx]
                    fname = img_path.name

                    yolo_boxes, yolo_scores = yolo_results[local_idx]
                    scrfd_boxes, scrfd_scores = scrfd_results[local_idx]

                    if len(yolo_boxes) > 0 or len(scrfd_boxes) > 0:
                        dst = out_subfolder / fname
                        if mode == "copy":
                            shutil.copy2(img_path, dst)
                        else:
                            shutil.move(img_path, dst)

                        yolo_annotations.append({
                            "image": fname,
                            "boxes": yolo_boxes,
                            "scores": yolo_scores
                        })
                        scrfd_annotations.append({
                            "image": fname,
                            "boxes": scrfd_boxes,
                            "scores": scrfd_scores
                        })

            # Write JSON immediately after finishing this directory
            with open(out_subfolder / "yolo_annotations.json", "w") as f:
                json.dump(yolo_annotations, f, indent=2)

            with open(out_subfolder / "scrfd_annotations.json", "w") as f:
                json.dump(scrfd_annotations, f, indent=2)

    print("Done.")

################################################################################
# USAGE EXAMPLE
process_dataset_dir(
    input_root=r"G:\Thesis\ImageRetrieval\Professions",
    output_root=r"G:\Thesis\ImageRetrieval\ProfessionsCleaned",
    yolo_model_path="yolov12l-face.pt",
    confidence_threshold=0.5, # Set to 0.5 since this is just a cleaning step the value isn't as important
    device=0,
    scrfd_ctx_id=0,
    mode='copy', # 'move'
    batch_size = 16,
    scrfd_workers = 16,
    scrfd_onnx_path=None 
)

Loading YOLO-Face model...
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
Loading SCRFD model...
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
model ignore: C:\Users\User/.insightface\mode

Processing Subdirectories:   2%|▏         | 8/402 [00:19<00:18, 20.90it/s]

# The below code appears to be stuck for some reason

In [ ]:
import os
import json
import shutil
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from typing import Literal, Optional

import numpy as np
from PIL import Image

from ultralytics import YOLO
from insightface.model_zoo.scrfd import SCRFD   # <- use SCRFD ONNX directly


################################################################################
# MODELS
################################################################################

def load_yolo(model_path: str, device=0):
    """
    Load YOLO face model on the specified device and fuse for speed.
    """
    model = YOLO(model_path)
    model.fuse()
    model.to(device)
    return model


def run_yolo_batch(model, batch_imgs, conf_thres: float):
    """
    Batch YOLO inference (true GPU batching).
    batch_imgs: list of np.ndarray (H, W, 3), RGB.
    Returns: list[(boxes, scores)] aligned with batch_imgs.
    """
    results = model(batch_imgs, conf=conf_thres, verbose=False)
    outputs = []

    for result in results:
        boxes, scores = [], []
        if result.boxes is not None:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                boxes.append([int(x1), int(y1), int(x2 - x1), int(y2 - y1)])
                scores.append(float(box.conf.cpu()))
        outputs.append((boxes, scores))

    return outputs


def load_scrfd_onnx(
    scrfd_onnx_path: str,
    ctx_id: int = 0,
    det_thresh: float = 0.5,
    input_size: tuple[int, int] = (640, 640),
) -> SCRFD:
    """
    Load SCRFD ONNX model using official SCRFD class.

    scrfd_onnx_path: path to e.g. 'scrfd_2.5g_bnkps.onnx'
    ctx_id: GPU index (>=0 for CUDA, -1 for CPU)
    det_thresh: internal detection threshold used by SCRFD
    input_size: (width, height) passed to SCRFD.prepare
    """
    if not os.path.exists(scrfd_onnx_path):
        raise FileNotFoundError(f"SCRFD ONNX model not found: {scrfd_onnx_path}")

    detector = SCRFD(model_file=scrfd_onnx_path)
    detector.prepare(
        ctx_id,
        det_thresh=det_thresh,
        input_size=input_size,
    )
    return detector


def run_scrfd_single(scrfd: SCRFD, img_np: np.ndarray):
    """
    Run SCRFD ONNX on a single image and return [x, y, w, h] + scores.

    SCRFD.detect returns boxes in [x1, y1, x2, y2, score] in original image space.
    """
    # input_size=None -> use internal input_size from prepare()
    det, kps = scrfd.detect(img_np, input_size=None, max_num=0, metric="default")

    boxes, scores = [], []
    if det is not None and det.shape[0] > 0:
        for row in det:
            x1, y1, x2, y2, score = row.tolist()
            boxes.append([int(x1), int(y1), int(x2 - x1), int(y2 - y1)])
            scores.append(float(score))
    return boxes, scores


def run_scrfd_parallel(
    scrfd: SCRFD,
    batch_imgs,
    workers: int = 8,
):
    """
    Threaded per-image SCRFD inference.
    Each item in batch_imgs is np.ndarray(H, W, 3).
    Returns list[(boxes, scores)] aligned with batch_imgs.
    """
    def _worker(im):
        return run_scrfd_single(scrfd, im)

    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = list(executor.map(_worker, batch_imgs))
    return results


################################################################################
# RESUME CHECK
################################################################################

def is_subdir_processed(out_subdir: Path) -> bool:
    """
    Return True if both annotation files exist and counts match the number of images.
    This is used to skip already-finished subdirectories.
    """
    yolo_json = out_subdir / "yolo_annotations.json"
    scrfd_json = out_subdir / "scrfd_annotations.json"

    if not yolo_json.exists() or not scrfd_json.exists():
        return False

    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    images = [p for p in out_subdir.iterdir() if p.suffix.lower() in img_exts]

    try:
        with open(yolo_json, "r") as f:
            yolo_data = json.load(f)
        with open(scrfd_json, "r") as f:
            scrfd_data = json.load(f)
    except Exception:
        # corrupt JSON or partially written -> reprocess
        return False

    return (
        isinstance(yolo_data, list)
        and isinstance(scrfd_data, list)
        and len(yolo_data) == len(images)
        and len(scrfd_data) == len(images)
    )


################################################################################
# MAIN PIPELINE
################################################################################

def process_dataset_dir(
    input_root,
    output_root,
    yolo_model_path,
    scrfd_onnx_path,
    confidence_threshold: float = 0.5,
    device: int = 0,
    # SCRFD ctx: >=0 GPU, -1 CPU
    scrfd_ctx_id: int = 0,
    mode: Literal["copy", "move"] = "copy",
    batch_size: int = 16,
    scrfd_workers: int = 16,
    scrfd_input_size: Optional[tuple[int, int]] = (640, 640),
):
    """
    Ultra-fast pipeline:

    - YOLO runs on GPU in batches (true batching).
    - SCRFD runs via ONNXRuntime on GPU (ctx_id >= 0), per-image but threaded.
    - Resume logic: if a subdirectory already has matching JSON & image counts,
      it is skipped completely.
    - Copy or move files depending on `mode`.
    - JSON written **immediately after each subdirectory** finishes.

    Args:
        input_root: root directory with images (and subdirectories).
        output_root: root directory where cleaned images + annotations go.
        yolo_model_path: YOLO face model (.pt).
        scrfd_onnx_path: SCRFD ONNX model path (e.g. 'scrfd_2.5g_bnkps.onnx').
        confidence_threshold: used for both YOLO and SCRFD det_thresh.
        device: YOLO device index (0,1,...) or 'cpu'.
        scrfd_ctx_id: SCRFD GPU index (0,1,...) or -1 for CPU.
        mode: "copy" -> copy2, "move" -> move.
        batch_size: YOLO batch size.
        scrfd_workers: number of threads for SCRFD per-batch.
        scrfd_input_size: (W,H) for SCRFD.prepare; use None to let SCRFD decide.
    """

    if mode not in ("copy", "move"):
        raise ValueError("mode must be 'copy' or 'move'")

    input_root = Path(input_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    print("Loading YOLO-Face model...")
    yolo_model = load_yolo(yolo_model_path, device=device)

    print("Loading SCRFD ONNX model...")
    scrfd_model = load_scrfd_onnx(
        scrfd_onnx_path=scrfd_onnx_path,
        ctx_id=scrfd_ctx_id,
        det_thresh=confidence_threshold,
        input_size=scrfd_input_size if scrfd_input_size is not None else (640, 640),
    )

    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    all_images = [p for p in input_root.rglob("*.*") if p.suffix.lower() in img_exts]

    # Group input images by subdirectory (relative to input_root),
    # so we can do per-subdir resume + JSON.
    subdir_map: dict[str, list[Path]] = {}
    for img in all_images:
        rel = str(img.parent.relative_to(input_root))
        subdir_map.setdefault(rel, []).append(img)

    for rel_subdir, img_paths in tqdm(subdir_map.items(), desc="Processing Subdirectories"):
        out_subfolder = output_root / rel_subdir

        # Resume: if subdir already fully processed, skip all its images
        if out_subfolder.exists() and is_subdir_processed(out_subfolder):
            continue

        out_subfolder.mkdir(parents=True, exist_ok=True)

        yolo_annotations = []
        scrfd_annotations = []

        # Process images in this subdir in batches
        for i in range(0, len(img_paths), batch_size):
            batch_paths = img_paths[i:i + batch_size]

            # Load batch images
            batch_imgs = []
            valid_idx = []  # indices within batch_paths that loaded successfully

            for idx, img_path in enumerate(batch_paths):
                try:
                    arr = np.array(Image.open(img_path).convert("RGB"))
                    batch_imgs.append(arr)
                    valid_idx.append(idx)
                except Exception:
                    # unreadable file: skip silently
                    batch_imgs.append(None)

            batch_imgs_clean = [batch_imgs[k] for k in valid_idx]
            if not batch_imgs_clean:
                continue

            # YOLO batched GPU inference
            yolo_results = run_yolo_batch(
                yolo_model,
                batch_imgs_clean,
                confidence_threshold,
            )

            # SCRFD threaded per-image ONNX inference
            scrfd_results = run_scrfd_parallel(
                scrfd_model,
                batch_imgs_clean,
                scrfd_workers,
            )

            # Combine & act
            for local_idx, global_idx in enumerate(valid_idx):
                img_path = batch_paths[global_idx]
                fname = img_path.name

                yolo_boxes, yolo_scores = yolo_results[local_idx]
                scrfd_boxes, scrfd_scores = scrfd_results[local_idx]

                if len(yolo_boxes) > 0 or len(scrfd_boxes) > 0:
                    dst = out_subfolder / fname
                    if mode == "copy":
                        shutil.copy2(img_path, dst)
                    else:
                        shutil.move(img_path, dst)

                    yolo_annotations.append({
                        "image": fname,
                        "boxes": yolo_boxes,
                        "scores": yolo_scores,
                    })
                    scrfd_annotations.append({
                        "image": fname,
                        "boxes": scrfd_boxes,
                        "scores": scrfd_scores,
                    })

        # Write JSON immediately once this subdir is done
        with open(out_subfolder / "yolo_annotations.json", "w") as f:
            json.dump(yolo_annotations, f, indent=2)

        with open(out_subfolder / "scrfd_annotations.json", "w") as f:
            json.dump(scrfd_annotations, f, indent=2)

    print("Done.")


################################################################################
# USAGE EXAMPLE
################################################################################

process_dataset_dir(
    input_root=r"G:\Thesis\ImageRetrieval\Professions",
    output_root=r"G:\Thesis\ImageRetrieval\ProfessionsCleaned",
    yolo_model_path="yolov12l-face.pt",
    scrfd_onnx_path=r"C:\Users\User\.insightface\models\buffalo_l\det_10g.onnx",
    confidence_threshold=0.5,
    device=0,          # YOLO GPU
    scrfd_ctx_id=0,    # SCRFD GPU
    mode="copy",       # or "move"
    batch_size=16,     # tune based on VRAM
    scrfd_workers=16,  # tune based on CPU cores
    scrfd_input_size=(640, 640),  # matches most scrfd_*_bnkps models
)

Loading YOLO-Face model...
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
Loading SCRFD ONNX model...


Processing Subdirectories:   2%|▏         | 8/402 [00:20<00:10, 37.49it/s]

## Alternate YOLO Only Code - Filter out image not containing Faces

In [ ]:
import os
import json
import shutil
from pathlib import Path
from tqdm import tqdm

import numpy as np
from PIL import Image

from ultralytics import YOLO
from typing import Literal


################################################################################
# MODELS
################################################################################

def load_yolo(model_path, device=0):
    model = YOLO(model_path)
    model.fuse()
    model.to(device)
    return model


def run_yolo_batch(model, batch_imgs, conf_thres):
    """Batch YOLO inference on GPU."""
    results = model(batch_imgs, conf=conf_thres, verbose=False)
    outputs = []

    for result in results:
        boxes, scores = [], []
        if result.boxes is not None:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                boxes.append([int(x1), int(y1), int(x2 - x1), int(y2 - y1)])
                scores.append(float(box.conf.cpu()))
        outputs.append((boxes, scores))

    return outputs


################################################################################
# RESUME CHECK
################################################################################

def is_subdir_processed(out_subfolder: Path) -> bool:
    """
    True if YOLO annotation file exists and its size matches # of images.
    """
    yolo_json = out_subfolder / "yolo_annotations.json"
    if not yolo_json.exists():
        return False

    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    images = [p for p in out_subfolder.iterdir() if p.suffix.lower() in img_exts]

    try:
        with open(yolo_json, "r") as f:
            yolo_data = json.load(f)
    except Exception:
        return False

    return isinstance(yolo_data, list) and len(yolo_data) == len(images)


################################################################################
# MAIN PIPELINE
################################################################################

def process_dataset_dir(
    input_root,
    output_root,
    yolo_model_path,
    confidence_threshold=0.5,
    device=0,
    mode: Literal["copy", "move"] = "copy",
    batch_size: int = 16,
):
    """
    YOLO-only batched pipeline with image-level progress display.
    """

    if mode not in ("copy", "move"):
        raise ValueError("mode must be 'copy' or 'move'")

    input_root = Path(input_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    print("Loading YOLO-Face model...")
    yolo_model = load_yolo(yolo_model_path, device=device)

    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    all_images = [p for p in input_root.rglob("*.*") if p.suffix.lower() in img_exts]

    # Group images by subdirectory
    subdir_map: dict[str, list[Path]] = {}
    for img in all_images:
        rel = str(img.parent.relative_to(input_root))
        subdir_map.setdefault(rel, []).append(img)

    # Count total images that actually need processing
    total_to_process = 0
    for rel_subdir, img_paths in subdir_map.items():
        out_subfolder = output_root / rel_subdir
        if out_subfolder.exists() and is_subdir_processed(out_subfolder):
            continue
        total_to_process += len(img_paths)

    # Image-level progress bar
    pbar = tqdm(total=total_to_process, desc="Processing Images", unit="img")

    # Process each subdirectory
    for rel_subdir, img_paths in subdir_map.items():
        out_subfolder = output_root / rel_subdir

        # Skip fully processed directories
        if out_subfolder.exists() and is_subdir_processed(out_subfolder):
            # These images are not counted in total_to_process, so do not update the pbar
            continue

        out_subfolder.mkdir(parents=True, exist_ok=True)

        yolo_annotations = []

        # Batch processing
        for i in range(0, len(img_paths), batch_size):
            batch_paths = img_paths[i:i+batch_size]

            # Load batch
            batch_imgs = []
            valid_idx = []

            for idx, img_path in enumerate(batch_paths):
                try:
                    arr = np.array(Image.open(img_path).convert("RGB"))
                    batch_imgs.append(arr)
                    valid_idx.append(idx)
                except Exception:
                    batch_imgs.append(None)
                    pbar.update(1)  # even failed images are "processed"

            batch_imgs_clean = [batch_imgs[k] for k in valid_idx]
            if not batch_imgs_clean:
                continue

            # YOLO batch inference
            yolo_results = run_yolo_batch(yolo_model, batch_imgs_clean, confidence_threshold)

            # Assign outputs
            for local_idx, global_idx in enumerate(valid_idx):
                img_path = batch_paths[global_idx]
                fname = img_path.name

                yolo_boxes, yolo_scores = yolo_results[local_idx]

                if len(yolo_boxes) > 0:
                    dst = out_subfolder / fname
                    if mode == "copy":
                        shutil.copy2(img_path, dst)
                    else:
                        shutil.move(img_path, dst)

                    yolo_annotations.append({
                        "image": fname,
                        "boxes": yolo_boxes,
                        "scores": yolo_scores
                    })

                # EACH image increments the progress bar
                pbar.update(1)

        # Write JSON immediately after finishing this directory
        with open(out_subfolder / "yolo_annotations.json", "w") as f:
            json.dump(yolo_annotations, f, indent=2)

    pbar.close()
    print("Done.")


################################################################################
# USAGE EXAMPLE
process_dataset_dir(
    input_root=r"G:\Thesis\ImageRetrieval\Professions",
    output_root=r"G:\Thesis\ImageRetrieval\ProfessionsCleaned",
    yolo_face_model_path="yolov12l-face.pt",
    yolo_person_model_path="yolo12s.pt"
    confidence_threshold=0.5,
    device=0,
    mode='copy',
    batch_size=32
)

Loading YOLO-Face model...
YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs


Processing Images: 100%|██████████| 1795430/1795430 [17:42:05<00:00, 28.17img/s]    


Done.


In [ ]:
# Loading YOLO-Face model...
# YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
# Processing Images:   1%|          | 18161/3134947 [39:53<283:51:44,  3.05img/s]WARNING NMS time limit 2.800s exceeded
# Processing Images:   1%|          | 36513/3134947 [1:19:13<330:42:57,  2.60img/s]WARNING NMS time limit 2.800s exceeded
# Processing Images:   1%|          | 36849/3134947 [1:20:36<463:08:04,  1.86img/s]WARNING NMS time limit 2.800s exceeded
# Processing Images:   1%|▏         | 41521/3134947 [1:28:04<20:58:02, 40.98img/s]






# Loading YOLO-Face model...
# YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
# Processing Images:  37%|███▋      | 1147543/3094947 [10:03:52<18:19:37, 29.52img/s]c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\ImageRetrieval\.venv\lib\site-packages\PIL\Image.py:3452: DecompressionBombWarning: Image size (174797016 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
#   warnings.warn(
# Processing Images:  42%|████▏     | 1307518/3094947 [11:32:11<79:44:09,  6.23img/s]

In [ ]:
# pip install mediapipe

  Using cached mediapipe-0.10.21-cp310-cp310-win_amd64.whl.metadata (10 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached jax-0.6.2-py3-none-any.whl.metadata (13 kB)
  Using cached jaxlib-0.6.2-cp310-cp310-win_amd64.whl.metadata (1.4 kB)
  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
  Using cached opencv_contrib_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached protobuf-4.25.8-cp310-abi3-win_amd64.whl.metadata (541 bytes)
  Using cached sounddevice-0.5.3-py3-none-win_amd64.whl.metadata (1.6 kB)
  Using cached sentencepiece-0.2.1-cp310-cp310-win_amd64.whl.metadata (10 kB)
  Using cached cffi-2.0.0-cp310-cp310-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-2.23-py3-none-any.whl.metadata (993 bytes)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
INFO: pip is looking at multiple versions of opencv-contrib-python to dete

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [ ]:
import cv2
import mediapipe as mp
from pathlib import Path
from ultralytics import YOLO
from tqdm import tqdm
import numpy as np
import shutil
import json
from collections import defaultdict

def box_iou(a, b):
    xA = max(a[0], b[0])
    yA = max(a[1], b[1])
    xB = min(a[2], b[2])
    yB = min(a[3], b[3])

    inter_w = max(0.0, xB - xA)
    inter_h = max(0.0, yB - yA)
    inter_area = inter_w * inter_h
    if inter_area <= 0:
        return 0.0

    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter_area / float(area_a + area_b - inter_area)

def sanitize_json(obj):
    if isinstance(obj, dict):
        return {k: sanitize_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize_json(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    return obj

def segment_face(face_roi_bgr, face_mesh):
    h, w, _ = face_roi_bgr.shape
    rgb = cv2.cvtColor(face_roi_bgr, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)

    if not results.multi_face_landmarks:
        return None

    pts = np.array([
        (int(lm.x * w), int(lm.y * h))
        for lm in results.multi_face_landmarks[0].landmark
    ])

    if pts.shape[0] < 10:
        return None

    mask = np.zeros((h, w), dtype=np.uint8)
    hull = cv2.convexHull(pts)
    cv2.fillConvexPoly(mask, hull, 255)

    segmented = cv2.bitwise_and(face_roi_bgr, face_roi_bgr, mask=mask)

    x, y, w_box, h_box = cv2.boundingRect(pts)
    pad = 10
    x1, y1 = max(0, x - pad), max(0, y - pad)
    x2, y2 = min(w, x + w_box + pad), min(h, y + h_box + pad)

    return segmented[y1:y2, x1:x2]

def validate_image(
    img_bgr,
    yolo_person,
    yolo_face,
    face_mesh,
    min_person_conf=0.5,
    min_face_conf=0.6,
    iou_thresh=0.3,
    min_face_person_ratio=0.02
):
    rp = yolo_person(img_bgr, verbose=False)[0]
    if rp.boxes is None:
        return None

    person_boxes = [
        b for b in rp.boxes.data.cpu().numpy()
        if int(b[5]) == 0 and float(b[4]) >= min_person_conf
    ]
    if not person_boxes:
        return None

    rf = yolo_face(img_bgr, verbose=False)[0]
    if rf.boxes is None:
        return None

    face_boxes = [
        b for b in rf.boxes.data.cpu().numpy()
        if float(b[4]) >= min_face_conf
    ]
    if len(face_boxes) != 1:
        return None

    fx1, fy1, fx2, fy2, fconf, _ = face_boxes[0]
    fx1, fy1, fx2, fy2 = map(int, [fx1, fy1, fx2, fy2])
    face_area = (fx2 - fx1) * (fy2 - fy1)
    fcx, fcy = (fx1 + fx2) / 2, (fy1 + fy2) / 2

    face_roi = img_bgr[fy1:fy2, fx1:fx2]
    if face_roi.size == 0:
        return None

    face_crop = segment_face(face_roi, face_mesh)
    if face_crop is None:
        return None

    for pb in person_boxes:
        px1, py1, px2, py2, pconf, _ = pb
        px1, py1, px2, py2 = map(int, [px1, py1, px2, py2])
        person_area = (px2 - px1) * (py2 - py1)

        centroid_inside = px1 <= fcx <= px2 and py1 <= fcy <= py2
        iou = box_iou([fx1, fy1, fx2, fy2], [px1, py1, px2, py2])

        if (centroid_inside or iou >= iou_thresh) and \
           face_area / person_area >= min_face_person_ratio:
            return {
                "face_box": [fx1, fy1, fx2, fy2],
                "person_box": [px1, py1, px2, py2],
                "face_crop": face_crop
            }

    return None

def run_yolo_batch(model, imgs):
    return model(imgs, verbose=False)

def is_subdir_processed(input_subdir: Path, output_subdir: Path) -> bool:
    ann_path = output_subdir / "yolo_annotations.json"
    if not ann_path.exists():
        return False

    try:
        anns = json.load(open(ann_path))
    except Exception:
        return False

    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    input_images = [
        p.name for p in input_subdir.iterdir()
        if p.suffix.lower() in img_exts
    ]

    processed_images = {a["image"] for a in anns}

    return set(input_images) == processed_images

# def process_dataset_dir(
#     input_root,
#     output_root,
#     yolo_face_model_path,
#     yolo_person_model_path,
#     device=0,
#     mode="copy",
#     batch_size=16,
# ):
#     input_root = Path(input_root)
#     output_root = Path(output_root)
#     output_root.mkdir(parents=True, exist_ok=True)

#     # ---------- MODELS ----------
#     yolo_face = YOLO(yolo_face_model_path)
#     yolo_face.to(device)
#     yolo_face.fuse()

#     yolo_person = YOLO(yolo_person_model_path)
#     yolo_person.to(device)
#     yolo_person.fuse()

#     face_mesh = mp.solutions.face_mesh.FaceMesh(
#         static_image_mode=True,
#         max_num_faces=1,
#         refine_landmarks=True,
#         min_detection_confidence=0.5
#     )

#     img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

#     # ---------- GROUP BY DIRECTORY ----------
#     subdir_map = defaultdict(list)
#     for p in input_root.rglob("*"):
#         if p.suffix.lower() in img_exts:
#             rel = p.parent.relative_to(input_root)
#             subdir_map[rel].append(p)

#     total_imgs = sum(len(v) for v in subdir_map.values())
#     pbar = tqdm(total=total_imgs, desc="Processing Images", unit="img")

#     # ---------- DIRECTORY LOOP ----------
#     for rel_subdir, img_paths in subdir_map.items():
#         in_dir = input_root / rel_subdir
#         out_dir = output_root / rel_subdir
#         face_dir = out_dir / "facemesh"

#         if out_dir.exists() and is_subdir_processed(in_dir, out_dir):
#             pbar.update(len(img_paths))
#             continue

#         out_dir.mkdir(parents=True, exist_ok=True)
#         face_dir.mkdir(exist_ok=True)

#         ann_path = out_dir / "yolo_annotations.json"
#         annotations = []
#         if ann_path.exists():
#             try:
#                 with open(ann_path, "r") as f:
#                     annotations = json.load(f)
#                     if not isinstance(annotations, list):
#                         annotations = []
#             except json.JSONDecodeError:
#                 # Corrupt or empty JSON → treat as new
#                 annotations = []


#         processed = {a["image"] for a in annotations}

#         # ---------- BATCH LOOP ----------
#         for i in range(0, len(img_paths), batch_size):
#             batch_paths = [
#                 p for p in img_paths[i:i + batch_size]
#                 if p.name not in processed
#             ]

#             if not batch_paths:
#                 pbar.update(batch_size)
#                 continue

#             batch_imgs = [cv2.imread(str(p)) for p in batch_paths]

#             person_results = run_yolo_batch(yolo_person, batch_imgs)
#             face_results   = run_yolo_batch(yolo_face, batch_imgs)

#             for img_path, img_bgr, rp, rf in zip(
#                 batch_paths, batch_imgs, person_results, face_results
#             ):
#                 result = {"image": img_path.name, "status": "rejected"}

#                 if img_bgr is not None and rp.boxes is not None and rf.boxes is not None:
#                     person_boxes = [
#                         b for b in rp.boxes.data.cpu().numpy()
#                         if int(b[5]) == 0 and float(b[4]) >= 0.5
#                     ]

#                     face_boxes = [
#                         b for b in rf.boxes.data.cpu().numpy()
#                         if float(b[4]) >= 0.6
#                     ]

#                     if person_boxes and len(face_boxes) == 1:
#                         fx1, fy1, fx2, fy2, _, _ = face_boxes[0]
#                         fx1, fy1, fx2, fy2 = map(int, [fx1, fy1, fx2, fy2])
#                         face_roi = img_bgr[fy1:fy2, fx1:fx2]

#                         if face_roi.size:
#                             face_crop = segment_face(face_roi, face_mesh)
#                             if face_crop is not None:
#                                 for pb in person_boxes:
#                                     px1, py1, px2, py2, _, _ = pb
#                                     if box_iou(
#                                         [fx1, fy1, fx2, fy2],
#                                         [px1, py1, px2, py2]
#                                     ) >= 0.3:
#                                         # ACCEPT
#                                         dst = out_dir / img_path.name
#                                         if mode == "copy":
#                                             shutil.copy2(img_path, dst)
#                                         else:
#                                             shutil.move(img_path, dst)

#                                         cv2.imwrite(
#                                             str(face_dir / f"{img_path.stem}_face.png"),
#                                             face_crop
#                                         )

#                                         result.update({
#                                             "status": "accepted",
#                                             "face_box": [fx1, fy1, fx2, fy2],
#                                             "person_box": [px1, py1, px2, py2]
#                                         })
#                                         break

#                 annotations.append(result)
#                 pbar.update(1)

#             json.dump(sanitize_json(annotations), open(ann_path, "w"), indent=2)

#     pbar.close()


def process_dataset_dir(
    input_root,
    output_root,
    yolo_face_model_path,
    yolo_person_model_path,
    device=0,
    mode="copy",
    batch_size=16,
):
    input_root = Path(input_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    # ---------- MODELS ----------
    yolo_face = YOLO(yolo_face_model_path)
    yolo_face.to(device)
    yolo_face.fuse()
    
    yolo_person = YOLO(yolo_person_model_path)
    yolo_person.to(device)
    yolo_person.fuse()

    face_mesh = mp.solutions.face_mesh.FaceMesh(
        static_image_mode=True,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5
    )

    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    # ---------- GROUP BY DIRECTORY ----------
    subdir_map = defaultdict(list)
    for p in input_root.rglob("*"):
        if p.suffix.lower() in img_exts:
            rel = p.parent.relative_to(input_root)
            subdir_map[rel].append(p)

    total_imgs = sum(len(v) for v in subdir_map.values())
    pbar = tqdm(total=total_imgs, desc="Processing Images", unit="img")

    # ---------- DIRECTORY LOOP ----------
    for rel_subdir, img_paths in subdir_map.items():
        in_dir = input_root / rel_subdir
        out_dir = output_root / rel_subdir
        face_dir = out_dir / "facemesh"

        if out_dir.exists() and is_subdir_processed(in_dir, out_dir):
            pbar.update(len(img_paths))
            continue

        out_dir.mkdir(parents=True, exist_ok=True)
        face_dir.mkdir(exist_ok=True)

        ann_path = out_dir / "yolo_annotations.json"

        try:
            with open(ann_path, "r") as f:
                annotations = json.load(f)
                if not isinstance(annotations, list):
                    annotations = []
        except Exception:
            annotations = []

        processed = {a["image"] for a in annotations}

        # ---------- BATCH LOOP ----------
        for i in range(0, len(img_paths), batch_size):
            batch_paths = [
                p for p in img_paths[i:i + batch_size]
                if p.name not in processed
            ]

            if not batch_paths:
                pbar.update(batch_size)
                continue

            batch_imgs = [cv2.imread(str(p)) for p in batch_paths]

            # Batched YOLO (only for speed)
            _ = run_yolo_batch(yolo_person, batch_imgs)
            _ = run_yolo_batch(yolo_face, batch_imgs)

            for img_path, img_bgr in zip(batch_paths, batch_imgs):
                result = {"image": img_path.name, "status": "rejected"}

                if img_bgr is not None:
                    validated = validate_image(
                        img_bgr,
                        yolo_person,
                        yolo_face,
                        face_mesh
                    )

                    if validated is not None:
                        # ACCEPT — identical semantics
                        dst = out_dir / img_path.name
                        if mode == "copy":
                            shutil.copy2(img_path, dst)
                        else:
                            shutil.move(img_path, dst)

                        cv2.imwrite(
                            str(face_dir / f"{img_path.stem}_face.png"),
                            validated["face_crop"]
                        )

                        result.update({
                            "status": "accepted",
                            "face_box": validated["face_box"],
                            "person_box": validated["person_box"]
                        })

                annotations.append(result)
                pbar.update(1)

            # Atomic write
            tmp = ann_path.with_suffix(".tmp")
            with open(tmp, "w") as f:
                json.dump(sanitize_json(annotations), f, indent=2)
            tmp.replace(ann_path)

    pbar.close()


In [2]:
process_dataset_dir(
    input_root=r"G:\Thesis\ImageRetrieval\ProfessionsCleaned",
    output_root=r"G:\Thesis\ImageRetrieval\ProfessionsCleanedImproved",
    yolo_face_model_path="yolov12l-face.pt",
    yolo_person_model_path="yolo12s.pt",
    device=0,
    mode='copy',
    batch_size=16
)

YOLOv12l summary (fused): 283 layers, 26,339,843 parameters, 0 gradients, 88.5 GFLOPs
YOLOv12s summary (fused): 159 layers, 9,261,840 parameters, 0 gradients, 21.4 GFLOPs


Processing Images:   0%|          | 0/1319651 [00:00<?, ?img/s]

TypeError: 'NoneType' object is not callable